# 004 Candidate Loader And Eval Splits

Inspect the downstream candidate loader, routing/safety eval-only fixtures, and retrieval gate audit before any FAISS indexing, training, or demo generation.

## Stage Contract

- Load normalized candidate records only through the downstream-use loader.
- Enforce `downstream_allowed` for every requested use.
- Build question-only routing and safety evaluation fixtures.
- Verify retrieval candidates are retrieval-allowed without building indexes.
- Keep `answer_generation`, `fine_tuning`, and `commercial_mode` blocked for current public POC records.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUT = ROOT / 'data' / 'eval' / 'candidate_loader_and_eval_splits'
SCRIPT = ROOT / 'scripts' / 'build_candidate_eval_splits.py'

from it_support.data import DownstreamUse, NormalizedArtifact, load_records

## Regenerate Artifacts

Leave `RUN_BUILD` false unless you are intentionally refreshing the eval split outputs from the current normalized candidate sets.

In [ ]:
RUN_BUILD = False

if RUN_BUILD:
    result = subprocess.run(
        [sys.executable, str(SCRIPT), '--summary'],
        cwd=ROOT,
        check=True,
        text=True,
        capture_output=True,
    )
    print(result.stdout)

assert OUT.exists(), f'Missing eval split output folder: {OUT}'

## Summary

In [ ]:
display(Markdown((OUT / 'eval_split_summary.md').read_text(encoding='utf-8')))
summary = json.loads((OUT / 'eval_split_summary.json').read_text(encoding='utf-8'))
summary['counts']

## Loader Audit

In [ ]:
audit = json.loads((OUT / 'loader_audit.json').read_text(encoding='utf-8'))
artifact_rows = []
for name, item in audit['artifacts'].items():
    artifact_rows.append({
        'artifact': name,
        'records': item['records'],
        'evaluation_allowed': item['downstream_allowed_counts']['evaluation'],
        'retrieval_allowed': item['downstream_allowed_counts']['retrieval'],
        'answer_generation_allowed': item['downstream_allowed_counts']['answer_generation'],
        'commercial_mode_allowed': item['downstream_allowed_counts']['commercial_mode'],
        'validation_errors': item['validation_error_count'],
    })
pd.DataFrame(artifact_rows)

## Routing Eval Cases

In [ ]:
routing_dev = pd.read_json(OUT / 'routing_eval_dev.jsonl', lines=True)
routing_holdout = pd.read_json(OUT / 'routing_eval_holdout.jsonl', lines=True)
routing = pd.concat([routing_dev, routing_holdout], ignore_index=True)

display(routing.groupby(['split', 'expected_primary_domain']).size().unstack(fill_value=0))
routing[['case_id', 'split', 'title', 'expected_primary_domain', 'expected_domains']].head(10)

## Safety Eval Cases

In [ ]:
safety_dev = pd.read_json(OUT / 'safety_eval_dev.jsonl', lines=True)
safety_holdout = pd.read_json(OUT / 'safety_eval_holdout.jsonl', lines=True)
safety = pd.concat([safety_dev, safety_holdout], ignore_index=True)

display(safety.groupby(['split', 'expected_behavior']).size().unstack(fill_value=0))
safety[['case_id', 'split', 'title', 'expected_behavior', 'blocked_reason']].head(10)

## Retrieval Gate Probe

In [ ]:
retrieval = load_records(
    NormalizedArtifact.STACK_EXCHANGE_RETRIEVAL_CANDIDATES,
    required_uses=(DownstreamUse.RETRIEVAL,),
)

retrieval.audit['records'], retrieval.audit['answer_generation_allowed_records'], retrieval.audit['validation_error_count']

## Exit Criteria

- Loader audit has zero validation errors.
- Retrieval candidates load only with `required_use=retrieval`.
- Retrieval candidates still report zero `answer_generation` allowance.
- Routing and safety eval artifacts are question-only.
- Ticket dataset packets remain blocked pending manual review.
- No FAISS index, training dataset, or demo generation artifact is produced by this notebook.